# Data Description

In [1]:
# libraries
import numpy as np
import pandas as pd
import pingouin as pg
import seaborn as sns
from matplotlib import pyplot as plt
import warnings

warnings.filterwarnings(action="ignore", category=RuntimeWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
# import data
df = pd.read_csv('datasets/datasets/aerofit_treadmill_data.csv')

## Data Types

In [3]:
# Data Types of Columns
def df_datatypes(df):
    df_desc = pd.DataFrame(df.dtypes.value_counts().reset_index())
    df_desc.columns = ['Data Type', 'Count']
    return df_desc.sort_values('Count', ascending=False)

df_datatypes(df)

,Data Type,Count
0,int64,6
1,str,3


## Missing Info

In [4]:
# DataFrame info
def get_df_info(df):
    df_info = pd.DataFrame()
    df_info['Column'] = df.columns.tolist()
    df_info['Data Type'] = df.dtypes.tolist()
    df_info['Non-Null Count'] = df.notnull().sum().tolist()
    df_info['# Null'] = df.isna().sum().tolist()
    df_info['Total Count'] = df.shape[0]
    df_info['% Null'] = np.round(df_info['# Null'] / df_info['Total Count'],2) * 100
    df_info['% Null'] = [str(n) + '%' for n in df_info['% Null']]
    return df_info

get_df_info(df)

,Column,Data Type,Non-Null Count,# Null,Total Count,% Null
0,Product,str,180,0,180,0.0%
1,Age,int64,180,0,180,0.0%
2,Gender,str,180,0,180,0.0%
3,Education,int64,180,0,180,0.0%
4,MaritalStatus,str,180,0,180,0.0%
5,Usage,int64,180,0,180,0.0%
6,Fitness,int64,180,0,180,0.0%
7,Income,int64,180,0,180,0.0%
8,Miles,int64,180,0,180,0.0%


In [5]:
# Numerical Info
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
get_df_info(df[numeric_columns])

,Column,Data Type,Non-Null Count,# Null,Total Count,% Null
0,Age,int64,180,0,180,0.0%
1,Education,int64,180,0,180,0.0%
2,Usage,int64,180,0,180,0.0%
3,Fitness,int64,180,0,180,0.0%
4,Income,int64,180,0,180,0.0%
5,Miles,int64,180,0,180,0.0%


In [6]:
# Categorical Info
categorical_columns = df.select_dtypes(exclude=np.number).columns.tolist()
get_df_info(df[categorical_columns])

,Column,Data Type,Non-Null Count,# Null,Total Count,% Null
0,Product,str,180,0,180,0.0%
1,Gender,str,180,0,180,0.0%
2,MaritalStatus,str,180,0,180,0.0%


## Categorical

In [7]:
def df_describe_categorical(df: pd.DataFrame) -> pd.DataFrame:
    # get categorical columns
    cat_df = df.select_dtypes(exclude=np.number)

    # return empty dataframe if empty
    if cat_df.empty:
        return pd.DataFrame()

    # get number of observations
    counts = cat_df.count()

    # get missing
    n_missing = cat_df.isna().sum()

    # get number of unique values
    uniques = cat_df.nunique()

    # get mode
    modes_df = cat_df.mode().iloc[0]

    # get top categories
    top_categories = modes_df.astype(str)

    # get top category frequency
    top_freqs = (cat_df == modes_df).sum()

    # get percentage of top category
    top_pcts = ((top_freqs / counts).round(2) * 100).astype(int).astype(str) + "%"

    # build dataframe
    summary_df = pd.DataFrame(
        {
            "Column": cat_df.columns,
            "Data Type": cat_df.dtypes,
            "Count": counts,
            "Missing Values": n_missing,
            "Unique Values": uniques,
            "Top Category": top_categories,
            "Top Category Frequency": top_freqs,
            "% Top Category": top_pcts,
        }
    ).reset_index(drop=True)

    return summary_df

df_describe_categorical(df)

,Column,Data Type,Count,Missing Values,Unique Values,Top Category,Top Category Frequency,% Top Category
0,Product,str,180,0,3,KP281,80,44%
1,Gender,str,180,0,2,Male,104,57%
2,MaritalStatus,str,180,0,2,Partnered,107,59%


## Numerical

In [8]:
# Numeric Description
def get_describe_numeric(df, precision=4):
    numeric_df = pd.DataFrame()
    numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
    numeric_df['Column'] = df[numeric_columns].columns.tolist()
    numeric_df['Data Type'] = df[numeric_columns].dtypes.tolist()
    numeric_df['Count'] = [df[col].count() for col in numeric_columns]
    numeric_df['Missing'] = [df[col].isna().sum() for col in numeric_columns]
    numeric_df['Mean'] = [df[col].mean() for col in numeric_columns]
    numeric_df['Std Dev'] = [df[col].std() for col in numeric_columns]
    numeric_df['Min'] = [df[col].min() for col in numeric_columns]
    numeric_df['25%'] = [np.nanpercentile(df[col], 25) for col in numeric_columns]
    numeric_df['50%'] = [np.nanpercentile(df[col], 50) for col in numeric_columns]
    numeric_df['75%'] = [np.nanpercentile(df[col], 75) for col in numeric_columns]
    numeric_df['Max'] = [df[col].max() for col in numeric_columns]
    
    # jarque_bera test for normality
    normality_df = pg.normality(df, method='jarque_bera').reset_index().rename({'index':'Column'}, axis='columns'); normality_df
    numeric_df = pd.merge(numeric_df, normality_df[['Column','normal', 'pval']], how='inner', on='Column')
    
    return numeric_df.round(precision)

get_describe_numeric(df)

,Column,Data Type,Count,Missing,Mean,Std Dev,Min,25%,50%,75%,Max,normal,pval
0,Age,int64,180,0,28.7889,6.9435,18,24.00,26.0,33.00,50,False,0.0000
1,Education,int64,180,0,15.5722,1.6171,12,14.00,16.0,16.00,21,False,0.0001
2,Usage,int64,180,0,3.4556,1.0848,2,3.00,3.0,4.00,7,False,0.0001
3,Fitness,int64,180,0,3.3111,0.9589,1,3.00,3.0,4.00,5,False,0.0266
4,Income,int64,180,0,53719.5778,16506.6842,29562,44058.75,50596.5,58668.00,104581,False,0.0000
5,Miles,int64,180,0,103.1944,51.8636,21,66.00,94.0,114.75,360,False,0.0000


### Correlogram

In [ ]:
# numerical - correlogram
def graph_correlogram(df):
    sns.set_theme(style="white") 
    # Compute the correlation matrix (use round to change decimals)
    corr = df.corr(numeric_only=True).round(2)
    # Generate a mask for the upper triangle
    mask = np.triu(np.ones_like(corr, dtype=bool))
    # Set up the matplotlib figure
    f, ax = plt.subplots(figsize=(11, 9))
    # Draw the heatmap with the mask and correct aspect ratio
    graph = sns.heatmap(
        corr, mask=mask, cmap='RdYlGn', vmax=.3, center=0, square=True,
        annot=True, linewidths=.5,
        cbar_kws={"shrink": .5}, annot_kws={"fontsize":15})
    return graph

graph_correlogram(df)
plt.show()

### Correlation Matrix

In [ ]:
# correlation matrix with p-values
def df_correlation_matrix(df):
    numerical = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f'Pearson Correlation Matrix with P-Values')
    print(f'[Coef in Btm Tri / p-Values in Up Tri]')
    print(f'*** for <0.001, ** for <0.01, * for <0.05')
    print(f'-----------------------------------------')
    return df[numerical].rcorr(method='pearson').round(3)

df_correlation_matrix(df)

# Histograms of Numerical

In [ ]:
# graph of all numeric data types
def graph_numeric_histograms(df):
    custom_params = {"axes.spines.right": False, "axes.spines.top": False}
    sns.set_theme(style="ticks", rc=custom_params)
    numeric_columns = df.select_dtypes(include=np.number)
    for col in numeric_columns:
        sns.histplot(df[col], kde=True, color='black')
        plt.title(f'Histogram of {col.title()}')
        plt.show()
    
graph_numeric_histograms(df)

# Analysis of Variation

## All vs Product

In [ ]:
def get_anova_df(target, df, is_target_cat=True):
    # libraries
    import warnings
    import pandas as pd
    import numpy as np
    import pingouin as pg
    
    if is_target_cat:
        # filter warnings
        warnings.simplefilter(action='ignore', category=FutureWarning)
        # get numeric features
        numerical_columns = df.select_dtypes(include=np.number).columns.tolist()
        # empty dataframe
        anova_df = pd.DataFrame()
        # loop over numeric features and run anova on cat
        for num in numerical_columns:
            new_row = df.anova(dv=num, between=target, detailed=False).round(4)
            new_row['Target'] = num
            new_row = new_row.rename(columns={'Target':'Source', 'Source':'Target'})
            anova_df = pd.concat([anova_df, new_row], axis='rows')
            anova_df['S.S. Diff'] = ['Yes' if p <= 0.05 else 'No' for p in anova_df['p-unc']]
        # reorder df
        anova_df = anova_df[['Target', 'Source', 'ddof1', 'ddof2', 'F', 'p-unc', 'np2', 'S.S. Diff']]
        return anova_df.round(4)
    else:
        # filter warnings
        warnings.simplefilter(action='ignore', category=FutureWarning)
        # get categorical features
        categorical_columns = df.select_dtypes(exclude=np.number).columns.tolist()
        # empty dataframe
        anova_df = pd.DataFrame()
        # loop over categorical features and run anova on num
        for cat in categorical_columns:
            new_row = df.anova(dv=target, between=cat, detailed=False)
            anova_df = pd.concat([anova_df, new_row], axis='rows')
            anova_df['S.S. Diff'] = ['Yes' if p <= 0.05 else 'No' for p in anova_df['p-unc']]
        anova_df['Target'] = target
        # reorder columns
        anova_df = anova_df[['Target', 'Source', 'ddof1', 'ddof2', 'F', 'p-unc', 'np2', 'S.S. Diff']]
        return anova_df.round(4)

In [ ]:
get_anova_df('Product', df, True)

## Pairwise Test

In [ ]:
def pw_comp(cat, num, data):
    # calculate effect size
    def get_eff_size(g):
        if np.abs(g) <= 0.2:
            return 'Small'
        elif np.abs(g) <= 0.5:
            return 'Medium'
        else:
            return 'Large'
    # graph
    graph = sns.boxplot(y=cat, x=num, data=data)
    plt.suptitle(f'Boxplot of {cat.title()} vs {num.title()}')
    plt.show()
    
    # display means
    display(pd.DataFrame(data.groupby(cat)[num].mean()))
    
    # add new columns
    aov_df = pg.pairwise_tukey(dv=num, between=cat, data=df).round(4)
    aov_df['S.S. Diff'] = ['Yes' if p <= 0.05 else 'No' for p in aov_df['p-tukey']]
    aov_df['Eff Size'] = aov_df['hedges'].apply(get_eff_size)
    
    display(aov_df)

In [ ]:
pw_comp('Product', 'Age', df)

In [ ]:
pw_comp('Product', 'Education', df)

In [ ]:
pw_comp('Product', 'Usage', df)

In [ ]:
pw_comp('Product', 'Fitness', df)

In [ ]:
pw_comp('Product', 'Income', df)

In [ ]:
pw_comp('Product', 'Miles', df)

# Chi Sqi Test

In [ ]:
def get_chi_sq_df(target, df):
    # filter warnings
    warnings.simplefilter(action='ignore', category=FutureWarning)
    categorical_columns = df.select_dtypes(exclude=np.number).columns.tolist()
    categorical_columns.remove(target)
    # empty df
    chi_sq_df = pd.DataFrame()
    # loop through df, add row for each pair
    for cat in categorical_columns:
        _,_,new_row = pg.chi2_independence(data=df, x=target, y=cat)
        new_row = new_row.loc[new_row['test']=='pearson']
        new_row['Target'] = target
        new_row['Categorical Feature'] = cat
        chi_sq_df = pd.concat([chi_sq_df, new_row], axis='rows')
        chi_sq_df['S.S. Diff'] = ['Yes' if p <= 0.05 else 'No' for p in chi_sq_df['pval']]
        chi_sq_df = chi_sq_df[['Target', 'Categorical Feature', 'test', 'chi2', 'dof', 'pval', 'cramer', 'power', 'S.S. Diff']]
    return chi_sq_df.round(4)

In [ ]:
get_chi_sq_df('Product', df)

In [ ]:
def chi_sq_info(x,y,data):
    # get info from chi sq test
    expected, observed, stats = pg.chi2_independence(data=data, x=x, y=y)
    def chi_sq_conc(p):
        if p <= 0.05:
            return 'Dependent'
        else:
            return 'Independent'
    # result of chi sq test
    stats['Conclusion'] = stats['pval'].apply(chi_sq_conc)
    
    # function for effect size
    def eff_size(c):
        if c <= 0.2:
            return 'Weak'
        elif c <= 0.6:
            return 'Moderate'
        else:
            return 'Strong'
    
    # add effect size column
    stats['Effect Size'] = stats['cramer'].apply(eff_size)
    
    # expected frequencies into a dataframe
    expected_melted = expected.reset_index().melt(id_vars=x)
    
    # observed frequencies into a dataframe
    observed_melted = observed.reset_index().melt(id_vars=x)
    
    # graph of expected frequencies
    sns.catplot(data=expected_melted, x=x, y="value", col=y, kind="bar")
    plt.suptitle('Expected Frequencies')
    
    # graph of observed frequencies
    sns.catplot(data=observed_melted, x=x, y="value", col=y, kind="bar")
    plt.suptitle('Observed Frequencies')
    plt.show()
    
    # expected frequencies contingency table
    print(f'The expected contingency table of frequencies.')
    display(expected.round(4))

    
    # observed frequencies contingency table
    print(f'The observed contingency table of frequencies.')
    display(observed.round(4))
    
    # test summary
    print(f'The test summary:')
    display(stats.loc[stats['test']=='pearson'].round(4))

In [ ]:
chi_sq_info('Product', 'Gender', df)

In [ ]:
chi_sq_info('Product', 'MaritalStatus', df)